# RSNA Knee Abnormality Detection — Dataset Discovery + EDA

Goal for this notebook:
1. Confirm the folder / file structure matches expectations
2. Understand label coverage (gold-labeled vs report-only studies)
3. Look at label distributions + co-occurrence
4. Explore report text (language, length)
5. Explore series-level metadata (plane, fluid-sensitive, fat-sat)
6. Spot-check raw DICOM headers + pixel data
7. Visualize a handful of slices



In [ ]:
import os
import glob
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# On Kaggle this is usually: /kaggle/input/rsna-knee-abnormality-detection
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_SERIES_CSV = DATA_DIR / "train_series.csv"
TEST_CSV = DATA_DIR / "test.csv"
TEST_SERIES_CSV = DATA_DIR / "test_series.csv"
SAMPLE_SUB_CSV = DATA_DIR / "sample_submission.csv"

TRAIN_SERIES_DIR = DATA_DIR / "train_series"
TEST_SERIES_DIR = DATA_DIR / "test_series"

for p in [TRAIN_CSV, TRAIN_SERIES_CSV, TEST_CSV, TEST_SERIES_CSV, SAMPLE_SUB_CSV]:
    print(f"{'OK ' if p.exists() else 'MISSING'} -> {p}")


## 1. Load CSVs and do basic sanity checks

In [ ]:
train = pd.read_csv(TRAIN_CSV)
train_series = pd.read_csv(TRAIN_SERIES_CSV)
test = pd.read_csv(TEST_CSV)
test_series = pd.read_csv(TEST_SERIES_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

print("train.csv:", train.shape)
print("train_series.csv:", train_series.shape)
print("test.csv:", test.shape)
print("test_series.csv:", test_series.shape)
print("sample_submission.csv:", sample_sub.shape)


In [ ]:
LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

train.head()


In [ ]:
train_series.head()


### 1a. Do CSV rows match folders on disk?

Confirms `train.csv` studies line up with `train_series/` folders, and that
`train_series.csv` rows line up with series subfolders. Catches path/schema
surprises early.


In [ ]:
train_study_folders = set(p.name for p in TRAIN_SERIES_DIR.iterdir() if p.is_dir())
csv_study_ids = set(train["StudyInstanceUID"])

print("Studies in train.csv        :", len(csv_study_ids))
print("Study folders in train_series/:", len(train_study_folders))
print("In CSV but no folder        :", len(csv_study_ids - train_study_folders))
print("Folder but not in CSV       :", len(train_study_folders - csv_study_ids))


In [ ]:
# Series-level check: for a sample of studies, confirm listed series folders exist
sample_studies = random.sample(sorted(train_study_folders), min(20, len(train_study_folders)))

missing_series = []
for study_id in sample_studies:
    study_dir = TRAIN_SERIES_DIR / study_id
    on_disk = set(p.name for p in study_dir.iterdir() if p.is_dir())
    expected = set(train_series.loc[train_series["StudyInstanceUID"] == study_id, "SeriesInstanceUID"])
    missing = expected - on_disk
    if missing:
        missing_series.append((study_id, missing))

print(f"Checked {len(sample_studies)} studies, {len(missing_series)} had missing series folders")
missing_series[:5]


## 2. Label coverage: gold-labeled vs report-only

This is the single most important number for the whole competition plan —
it tells us how much of the training set needs silver labels derived
from the report text.


In [ ]:
# A study is "gold-labeled" if all 12 label columns are non-null.
# Adjust this logic if partially-labeled rows turn out to exist.
is_fully_labeled = train[LABEL_COLS].notna().all(axis=1)
is_any_labeled = train[LABEL_COLS].notna().any(axis=1)

n_total = len(train)
n_fully_labeled = is_fully_labeled.sum()
n_partially_labeled = (is_any_labeled & ~is_fully_labeled).sum()
n_unlabeled = (~is_any_labeled).sum()

print(f"Total studies          : {n_total}")
print(f"Fully labeled          : {n_fully_labeled} ({n_fully_labeled/n_total:.1%})")
print(f"Partially labeled      : {n_partially_labeled} ({n_partially_labeled/n_total:.1%})")
print(f"No labels (report only): {n_unlabeled} ({n_unlabeled/n_total:.1%})")


In [ ]:
# Does every row have a report, regardless of label status?
has_report = train["Report"].notna() & (train["Report"].str.strip() != "")
print(f"Studies with non-empty Report text: {has_report.sum()} / {n_total} ({has_report.mean():.1%})")
print(f"Fully-labeled studies WITHOUT a report: {(is_fully_labeled & ~has_report).sum()}")
print(f"Unlabeled studies WITHOUT a report (dead weight, unusable): {(~is_any_labeled & ~has_report).sum()}")


## 3. Label distribution (gold-labeled subset only)

In [ ]:
gold = train[is_fully_labeled].copy()

label_prevalence = gold[LABEL_COLS].mean().sort_values(ascending=False)
print("Positive rate per label (gold-labeled studies):")
print(label_prevalence)

fig, ax = plt.subplots(figsize=(9, 5))
label_prevalence.plot(kind="barh", ax=ax)
ax.set_xlabel("Positive rate")
ax.set_title(f"Label prevalence (n={len(gold)} gold-labeled studies)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# How many positive labels does a study typically have? (multi-label density)
n_positive_per_study = gold[LABEL_COLS].sum(axis=1)
print(n_positive_per_study.describe())

fig, ax = plt.subplots(figsize=(7, 4))
n_positive_per_study.value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_xlabel("Number of positive findings in a study")
ax.set_ylabel("Count of studies")
ax.set_title("How many findings co-occur per study")
plt.tight_layout()
plt.show()


In [ ]:
# Co-occurrence matrix between labels (correlation)
import seaborn as sns  # if unavailable on Kaggle image, pip install seaborn

corr = gold[LABEL_COLS].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Label co-occurrence (correlation)")
plt.tight_layout()
plt.show()


## 4. Report text exploration

Language mix and report length matter a lot for Week 2-3 label-extraction
work (regex/keyword pass) and the later LLM-based extraction pass.


In [ ]:
train["report_len_chars"] = train["Report"].fillna("").str.len()
train["report_len_words"] = train["Report"].fillna("").str.split().str.len()

print(train["report_len_words"].describe())

fig, ax = plt.subplots(figsize=(7, 4))
train.loc[has_report, "report_len_words"].hist(bins=50, ax=ax)
ax.set_xlabel("Report length (words)")
ax.set_title("Report length distribution")
plt.tight_layout()
plt.show()


In [ ]:
# Rough language detection on a sample (install if needed: pip install langdetect)
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

sample_reports = train.loc[has_report, "Report"].sample(min(300, has_report.sum()), random_state=42)

def safe_detect(text):
    try:
        return detect(text)
    except Exception:
        return "unknown"

langs = sample_reports.apply(safe_detect)
lang_counts = langs.value_counts()
print(lang_counts)

fig, ax = plt.subplots(figsize=(7, 4))
lang_counts.plot(kind="bar", ax=ax)
ax.set_title(f"Detected report language (sample of {len(sample_reports)})")
plt.tight_layout()
plt.show()


In [ ]:
# Does report length or presence differ between gold-labeled and unlabeled studies?
# (checks for sampling bias in which studies got gold labels)
comparison = pd.DataFrame({
    "gold_labeled": train.loc[is_fully_labeled, "report_len_words"].describe(),
    "unlabeled": train.loc[~is_any_labeled, "report_len_words"].describe(),
})
comparison


In [ ]:
# Eyeball a few raw reports
for i, row in train.loc[has_report].sample(3, random_state=1).iterrows():
    print("=" * 80)
    print("StudyInstanceUID:", row["StudyInstanceUID"])
    print("Labeled:", bool(is_fully_labeled.loc[i]))
    print(row["Report"][:800])


## 5. Series-level metadata (`train_series.csv`)

In [ ]:
print("Anatomical_Plane distribution:")
print(train_series["Anatomical_Plane"].value_counts(dropna=False))

print("\nFluid_Sensitive distribution:")
print(train_series["Fluid_Sensitive"].value_counts(dropna=False))

print("\nFat_Suppression distribution:")
print(train_series["Fat_Suppression"].value_counts(dropna=False))


In [ ]:
# Cross-tab of plane x fluid-sensitive x fat-sat -> the "vocabulary" of sequence types
seq_type_counts = (
    train_series
    .groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
seq_type_counts


In [ ]:
# Series per study distribution
series_per_study = train_series.groupby("StudyInstanceUID").size()
print(series_per_study.describe())

fig, ax = plt.subplots(figsize=(7, 4))
series_per_study.hist(bins=30, ax=ax)
ax.set_xlabel("Number of series per study")
ax.set_title("Series-per-study distribution")
plt.tight_layout()
plt.show()


In [ ]:
# Do all studies have at least one sagittal series? Coronal? This matters for
# ACL (sagittal) / meniscus (sagittal+coronal) modeling decisions later.
plane_presence = (
    train_series
    .assign(has_plane=1)
    .pivot_table(index="StudyInstanceUID", columns="Anatomical_Plane", values="has_plane",
                 aggfunc="max", fill_value=0)
)
print("Fraction of studies with each plane present at least once:")
print(plane_presence.mean())


## 6. DICOM header + pixel spot-check

Pulling a small random sample of raw `.dcm` files to look at transfer syntax,
pixel array shape, intensity range, and orientation. This is NOT meant to
process the full dataset — just enough to inform Week 2 preprocessing design.


In [ ]:
import pydicom

def sample_dcm_paths(n=15, seed=0):
    rng = random.Random(seed)
    study_ids = rng.sample(sorted(train_study_folders), min(n, len(train_study_folders)))
    paths = []
    for sid in study_ids:
        study_dir = TRAIN_SERIES_DIR / sid
        series_dirs = [p for p in study_dir.iterdir() if p.is_dir()]
        if not series_dirs:
            continue
        series_dir = rng.choice(series_dirs)
        dcm_files = list(series_dir.glob("*.dcm"))
        if not dcm_files:
            continue
        paths.append(rng.choice(dcm_files))
    return paths

dcm_paths = sample_dcm_paths(15)
print(f"Sampled {len(dcm_paths)} DICOM files")


In [ ]:
rows = []
for p in dcm_paths:
    ds = pydicom.dcmread(p)
    arr = ds.pixel_array
    rows.append({
        "path": str(p.relative_to(TRAIN_SERIES_DIR)),
        "TransferSyntaxUID": str(ds.file_meta.TransferSyntaxUID) if hasattr(ds, "file_meta") else None,
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "min": arr.min(),
        "max": arr.max(),
        "PixelSpacing": getattr(ds, "PixelSpacing", None),
        "SliceThickness": getattr(ds, "SliceThickness", None),
        "Modality": getattr(ds, "Modality", None),
    })

dcm_info = pd.DataFrame(rows)
dcm_info


In [ ]:
print("Unique transfer syntaxes seen in this sample:")
print(dcm_info["TransferSyntaxUID"].value_counts())

print("\nIntensity range summary:")
print(dcm_info[["min", "max"]].describe())


## 7. Visual sanity check: render a few slices across plane/sequence types

In [ ]:
def load_middle_slice(series_dir):
    dcm_files = sorted(series_dir.glob("*.dcm"))
    if not dcm_files:
        return None
    mid = dcm_files[len(dcm_files) // 2]
    ds = pydicom.dcmread(mid)
    return ds.pixel_array

# Grab one series per Anatomical_Plane to eyeball
planes_seen = {}
for _, row in train_series.sample(frac=1, random_state=3).iterrows():
    plane = row["Anatomical_Plane"]
    if plane in planes_seen:
        continue
    series_dir = TRAIN_SERIES_DIR / row["StudyInstanceUID"] / row["SeriesInstanceUID"]
    if series_dir.exists():
        planes_seen[plane] = series_dir
    if len(planes_seen) >= 3:
        break

fig, axes = plt.subplots(1, len(planes_seen), figsize=(5 * len(planes_seen), 5))
if len(planes_seen) == 1:
    axes = [axes]
for ax, (plane, series_dir) in zip(axes, planes_seen.items()):
    img = load_middle_slice(series_dir)
    if img is not None:
        ax.imshow(img, cmap="gray")
    ax.set_title(plane)
    ax.axis("off")
plt.tight_layout()
plt.show()
